In [10]:
import pandas as pd
import os

accounts_path = "../data/processed/accounts_clean.csv"
transactions_path = "../data/processed/transactions_clean.csv"

accounts = pd.read_csv(accounts_path)
transactions = pd.read_csv(transactions_path)

In [11]:
print("Accounts:", accounts.shape)
print("Transactions:", transactions.shape)

Accounts: (518581, 5)
Transactions: (2505191, 11)


In [12]:
print("Accounts columns:")
print(accounts.columns.tolist())

print("\nTransactions columns:")
print(transactions.columns.tolist())

Accounts columns:
['Bank Name', 'Bank ID', 'Account Number', 'Entity ID', 'Entity Name']

Transactions columns:
['Timestamp', 'From Bank', 'Sender_Account', 'To Bank', 'Receiver_Account', 'Amount Received', 'Receiving Currency', 'Amount Paid', 'Payment Currency', 'Payment Format', 'Is Laundering']


In [13]:
print("Accounts missing values:")
print(accounts.isna().sum())

print("\nTransactions missing values:")
print(transactions.isna().sum())

Accounts missing values:
Bank Name         0
Bank ID           0
Account Number    0
Entity ID         0
Entity Name       0
dtype: int64

Transactions missing values:
Timestamp             0
From Bank             0
Sender_Account        0
To Bank               0
Receiver_Account      0
Amount Received       0
Receiving Currency    0
Amount Paid           0
Payment Currency      0
Payment Format        0
Is Laundering         0
dtype: int64


In [14]:
pip install sqlalchemy psycopg2-binary python-dotenv

Note: you may need to restart the kernel to use updated packages.


In [15]:
from sqlalchemy import create_engine
from dotenv import load_dotenv
import os

In [16]:
load_dotenv()

DATABASE_URL = os.getenv("DATABASE_URL")

In [17]:
if DATABASE_URL:
    print("Database connection string loaded.")
else:
    print("DATABASE_URL not found.")

Database connection string loaded.


In [18]:
engine = create_engine(DATABASE_URL)

In [19]:
from sqlalchemy import text

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("PostgreSQL connection successful.")

PostgreSQL connection successful.


In [23]:
accounts.to_sql(
    "accounts",
    engine,
    if_exists="replace",
    index=False
)

581

In [21]:
transactions.to_sql(
    "transactions",
    engine,
    if_exists="replace",
    index=False
)

191

In [24]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT table_name
            FROM information_schema.tables
            WHERE table_schema = 'public'
        """)
    )

    for row in result:
        print(row[0])

transactions
accounts


In [25]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM accounts")
    )
    
    print("Accounts in PostgreSQL:", result.scalar())

Accounts in PostgreSQL: 518581


In [26]:
with engine.connect() as connection:
    result = connection.execute(
        text("SELECT COUNT(*) FROM transactions")
    )
    
    print("Transactions in PostgreSQL:", result.scalar())

Transactions in PostgreSQL: 2505191


In [27]:
print("Expected accounts:", len(accounts))
print("Expected transactions:", len(transactions))

Expected accounts: 518581
Expected transactions: 2505191


In [28]:
with engine.connect() as connection:
    result = connection.execute(
        text("""
            SELECT column_name
            FROM information_schema.columns
            WHERE table_name = 'transactions'
            ORDER BY ordinal_position
        """)
    )

    for row in result:
        print(row[0])

Timestamp
From Bank
Sender_Account
To Bank
Receiver_Account
Amount Received
Receiving Currency
Amount Paid
Payment Currency
Payment Format
Is Laundering


In [30]:
print("===== PART C SUMMARY =====")

print(f"Processed Accounts Rows      : {len(accounts):,}")
print(f"Processed Transactions Rows  : {len(transactions):,}")

print("\nPostgreSQL tables:")
print("✓ accounts")
print("✓ transactions")

print("\nDatabase column transformation:")
print("✓ Account → Sender_Account")
print("✓ Account.1 → Receiver_Account")

print("\nPostgreSQL hosting:")
print("✓ Data hosted in Neon PostgreSQL")

===== PART C SUMMARY =====
Processed Accounts Rows      : 518,581
Processed Transactions Rows  : 2,505,191

PostgreSQL tables:
✓ accounts
✓ transactions

Database column transformation:
✓ Account → Sender_Account
✓ Account.1 → Receiver_Account

PostgreSQL hosting:
✓ Data hosted in Neon PostgreSQL
